# Lab 4 — Transform Coding: Representations and Quantization

**Student laboratory · Topic 3: Time–frequency representations**

The lab connects the course slides with the example notebooks `dft-dct.ipynb`, `stft-mdct.ipynb` and `MCDT-quantization.ipynb`. Its focus is **what the representations tell us, why the MDCT is useful for audio coding, and what quantization changes**—not reimplementing transforms.

**Learning outcomes**
1. Compare DFT and DCT-IV basis coefficients and check invertibility.
2. Interpret time–frequency representations of tonal and transient audio, and explain MDCT critical sampling.
3. Measure and listen to MDCT quantization errors, distinguish coefficients from encoded bits, and explain the importance of bit allocation.

The supplied transform functions are tools. **Complete the student questions and your observations in your own Markdown cells.** The notebook works with WAV files in `../sounds/` and provides a deterministic synthetic signal only when files are unavailable.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import wavfile
from scipy.fft import rfft, irfft, rfftfreq, dct, idct
from pathlib import Path
from IPython.display import Audio, display

np.set_printoptions(precision=4, suppress=True)
EPS = np.finfo(float).tiny


def load_mono(path, fallback="harmonic", fs_fallback=16000, duration=2.5):
    """Load a WAV, scale PCM correctly, and mix stereo down to mono.

    Return a deterministic synthetic sound if the course recording is absent.
    No resampling is performed: always use the returned sampling rate.
    """
    path = Path(path)
    if path.is_file():
        fs, samples = wavfile.read(path)
        if np.issubdtype(samples.dtype, np.integer):
            if np.issubdtype(samples.dtype, np.unsignedinteger):
                info = np.iinfo(samples.dtype)
                x = (samples.astype(float) - (info.max + 1)/2) / ((info.max + 1)/2)
            else:
                info = np.iinfo(samples.dtype)
                x = samples.astype(float) / max(abs(info.min), abs(info.max))
        else:
            x = samples.astype(float)
        if x.ndim == 2:
            x = x.mean(axis=1)
        return x, int(fs)
    fs = fs_fallback
    t = np.arange(int(duration*fs))/fs
    if fallback == "sine":
        x = 0.65*np.sin(2*np.pi*440*t)
    else:
        # Harmonic, amplitude-modulated synthetic instrument-like sound.
        x = sum((0.55/h)*np.sin(2*np.pi*h*220*t + h*.2) for h in range(1, 9))
        x *= 0.7 + 0.25*np.sin(2*np.pi*2.2*t)
    print(f"Using synthetic '{fallback}' sound ({duration:g} s, {fs} Hz): {path} not found.")
    return x, fs


def listen(x, fs, label):
    """Playback with a fixed gain. Do not normalize clips separately."""
    print(label)
    display(Audio(np.asarray(x, dtype=float)*playback_gain, rate=int(fs), normalize=False))


def report_error(reference, reconstructed):
    """Relative RMS error on equal-length arrays; 0 means exact reconstruction."""
    assert len(reference) == len(reconstructed)
    return np.linalg.norm(reference-reconstructed)/max(np.linalg.norm(reference), EPS)


## Part 1 — DFT and DCT-IV of a short sound

An $M$-point DFT uses complex sinusoidal basis functions. The DCT-IV represents the same $M$ real sample values using $M$ real, signed cosine coefficients. Both transforms are invertible and **neither compresses audio by itself**. See slides on DFT, DCT-IV, coefficient timing, and the `dft-dct` example.

Run the comparison, then replace the sine-wave recording with a complex instrumental sound. **Do not compare raw coefficient indices as if they referred to identical basis frequencies.**

In [ ]:
# Compare transforms on one frame of a sine wave.
x_sine, fs_sine = load_mono("../sounds/sinewave.wav", "sine")
M = 512
start = min(2000, max(0, len(x_sine)-M))
frame = x_sine[start:start+M]

X_dft = rfft(frame, norm="ortho")
X_dct = dct(frame, type=4, norm="ortho")
frequencies_dft = rfftfreq(M, 1/fs_sine)
# DCT-IV basis frequencies in Hz (not the same grid as DFT bins).
frequencies_dct = (np.arange(M)+0.5)*fs_sine/(2*M)

fig, ax = plt.subplots(3, 1, figsize=(10, 8), constrained_layout=True)
ax[0].plot(np.arange(M)/fs_sine, frame)
ax[0].set(xlabel="Time (s)", ylabel="Amplitude", title="Input frame")
ax[1].plot(frequencies_dft, 20*np.log10(np.maximum(np.abs(X_dft), 1e-12)))
ax[1].set(xlabel="DFT frequency (Hz)", ylabel="Magnitude (dB)", title="DFT magnitude (orthonormal)")
ax[2].plot(frequencies_dct, 20*np.log10(np.maximum(np.abs(X_dct), 1e-12)))
ax[2].set(xlabel="DCT-IV basis frequency (Hz)", ylabel="Coefficient magnitude (dB)", title="DCT-IV magnitude")
plt.show()

x_from_dft = irfft(X_dft, n=M, norm="ortho")
x_from_dct = idct(X_dct, type=4, norm="ortho")
print(f"DFT reconstruction error (max abs): {np.max(np.abs(frame-x_from_dft)):.2e}")
print(f"DCT-IV reconstruction error (max abs): {np.max(np.abs(frame-x_from_dct)):.2e}")


**Questions — Part 1**

1. Compare the strongest coefficients for the sine wave and an instrumental sound. What does each transform emphasize?
2. How does DCT-IV represent waveform timing even though its coefficients are real? (Hint: the signs and weights change when the waveform is shifted.)
3. Report the maximum inverse-transform errors. Does a perfectly invertible transform achieve data compression? Explain.

*Write your answers below.*

## Part 2 — Time–frequency resolution and critical sampling

**Motivation:** A long window helps distinguish nearby stable frequencies; a short window helps localize attacks. Compare the STFTs of **the same sound** with two window lengths. This is a practical link to the class slides on time–frequency resolution, transient handling, and the STFT–MDCT comparison.

The MDCT implementation is provided below only to support **reconstruction and coefficient accounting** in Part 3. Its internal matrix construction is **not an exercise**.

In the MDCT, a block of $2M$ samples produces $M$ coefficients and advances $M$ samples. Thus there is one coefficient per new input sample in steady state. With a suitable window, overlapping inverse frames cancel time-domain aliasing (TDAC).

In [ ]:
from scipy.signal import stft

# Choose an instrument recording, or use the deterministic harmonic/transient fallback.
x, fs = load_mono('../sounds/487465-trumpet.wav')
x = x[:min(len(x), 3*fs)]

# Insert an optional short transient in the fallback only if a recording is absent.
if not Path('../sounds/487465-trumpet.wav').is_file():
    x = x.copy()
    event = min(len(x)//2, int(1.0*fs))
    x[event:event+min(16, len(x)-event)] += 0.5*np.hanning(min(16, len(x)-event))

fig, axes = plt.subplots(2, 1, figsize=(11, 6), sharex=True, constrained_layout=True)
for ax, frame_size in zip(axes, (256, 2048)):
    if len(x) < frame_size:
        frame_size = min(256, len(x))
    hop = frame_size // 4
    f, t, Z = stft(x, fs=fs, window='hann', nperseg=frame_size,
                   noverlap=frame_size-hop, boundary='zeros', padded=True)
    db = 20*np.log10(np.maximum(np.abs(Z), 1e-10))
    ax.pcolormesh(t, f, db, shading='auto', vmin=db.max()-75, vmax=db.max())
    ax.set(title=f'STFT: window {frame_size} samples ({1000*frame_size/fs:.1f} ms), hop {hop}',
           ylabel='Frequency (Hz)', ylim=(0, min(4500, fs/2)))
axes[-1].set_xlabel('Time (s)')
plt.show()

# Listen once: individual plots do not alter the audio.
playback_gain = 0.85 / max(1.0, np.max(np.abs(x)))
listen(x, fs, 'Input sound')

In [ ]:
# Construct an orthonormal MDCT cosine basis and Princen-Bradley sine window.
# Basis matrix has shape (2*M, M), NOT (M, M).
M_mdct = 128
n = np.arange(2*M_mdct)[:, None]
k = np.arange(M_mdct)[None, :]
basis = np.sqrt(2/M_mdct)*np.cos(
    np.pi/M_mdct*(n + 0.5 + M_mdct/2)*(k + 0.5)
)
window = np.sin(np.pi*(np.arange(2*M_mdct)+0.5)/(2*M_mdct))
assert np.allclose(window[:M_mdct]**2 + window[M_mdct:]**2, 1.0)


def mdct_analyze(x):
    """Windowed MDCT with 50% overlap; pad boundaries for accurate reconstruction."""
    x = np.asarray(x, dtype=float)
    # One M-sample boundary pad plus enough at the end for complete frames.
    extra = (-len(x)) % M_mdct
    padded = np.pad(x, (M_mdct, M_mdct+extra))
    coeffs = np.stack([
        (padded[start:start+2*M_mdct]*window) @ basis
        for start in range(0, len(padded)-2*M_mdct+1, M_mdct)
    ])
    return coeffs, len(x)


def mdct_synthesize(coeffs, original_length):
    """IMDCT followed by sine window and overlap-add (TDAC)."""
    output = np.zeros((len(coeffs)+1)*M_mdct, dtype=float)
    for i, c in enumerate(coeffs):
        start = i*M_mdct
        output[start:start+2*M_mdct] += (basis @ c)*window
    return output[M_mdct:M_mdct+original_length]




In [ ]:
# MDCT representation and perfect reconstruction before quantization.
X_mdct, original_length = mdct_analyze(x)
y_mdct = mdct_synthesize(X_mdct, original_length)

print(f'Input samples: {len(x)}')
print(f'MDCT coefficients: {X_mdct.size} (including boundary padding)')
print(f'Frames: {len(X_mdct)}, coefficients per frame: {M_mdct}')
print(f'New samples per frame (hop): {M_mdct}')
print(f'MDCT relative RMS error before quantization: {report_error(x, y_mdct):.3e}')
print(f'Coefficient/sample ratio including boundary padding: {X_mdct.size/len(x):.3f}')

# MDCT coefficients are signed; the absolute values are plotted only for visualization.
plt.figure(figsize=(11, 3.5))
plt.imshow(20*np.log10(np.maximum(np.abs(X_mdct.T), 1e-9)),
           origin='lower', aspect='auto', cmap='magma')
plt.title('MDCT absolute coefficient magnitudes (dB, display only)')
plt.xlabel('Frame'); plt.ylabel('MDCT coefficient index'); plt.colorbar(label='dB')
plt.show()

**Questions — Part 2**

1. Which STFT window better separates stable nearby frequencies? Which better localizes the onset of a short event? Explain the time–frequency trade-off.
2. For the MDCT, identify the analysis length, hop size, and coefficients per frame. Explain **why overlapping input blocks do not necessarily mean a redundant coefficient rate**.
3. Why does the coefficient/sample count slightly exceed 1 for a finite recording? Explain the role of boundary padding.
4. Is reconstruction essentially exact before quantization? Explain how TDAC makes this possible without implementing it.

*Write your answers below.*

## Part 3 — MDCT quantization: distortion and nominal bitrate

A transform alone is invertible. **Quantization**, not the transform, introduces information loss. Use the **same MDCT coefficients** for global and per-frequency-bin quantization. Compare these against time-domain quantization at the same nominal bit depth.

The global quantizer uses a single data-dependent range; the per-bin quantizer uses separate ranges. In a real codec those ranges would need to be represented in the bitstream (or otherwise known by the decoder), so equal nominal bits per value do **not** guarantee equal full-file bitrate. These are illustrative quantizers, not perceptual bit-allocation algorithms.

In [ ]:
def uniform_quantize(values, bits, axis=None):
    """Mid-tread-like uniform min/max quantization with 2**bits discrete levels.

    axis=None: one global range; axis=0: one range per column (MDCT band).
    A zero-range array is returned unchanged to avoid division by zero.
    """
    if bits < 1:
        raise ValueError("bits must be at least 1")
    values = np.asarray(values, dtype=float)
    lo = values.min(axis=axis, keepdims=True)
    hi = values.max(axis=axis, keepdims=True)
    span = hi-lo
    levels = 2**bits - 1
    step = np.where(span > 0, span/levels, 1.0)
    indices = np.clip(np.rint((values-lo)/step), 0, levels)
    return lo+indices*step


bits = 6   # Change this value when studying quantization
X_mdct, length = mdct_analyze(x)
q_per_band = uniform_quantize(X_mdct, bits, axis=0)
q_global = uniform_quantize(X_mdct, bits, axis=None)
y_per_band = mdct_synthesize(q_per_band, length)
y_global = mdct_synthesize(q_global, length)
y_pcm = uniform_quantize(x, bits, axis=None)

for name, candidate in [
    ("MDCT, per-band", y_per_band),
    ("MDCT, global", y_global),
    ("Time-domain PCM", y_pcm),
]:
    print(f"{name}: relative RMS error = {report_error(x, candidate):.4f}")

playback_gain = 0.9 / max(1.0, *(np.max(np.abs(a)) for a in [x, y_per_band, y_global, y_pcm]))
listen(x, fs, "Original")
listen(y_per_band, fs, f"MDCT per-band, {bits} bits/coefficient")
listen(y_global, fs, f"MDCT global, {bits} bits/coefficient")
listen(y_pcm, fs, f"Time-domain quantization, {bits} bits/sample")


In [ ]:
# Explore distortion as a function of uniform quantization depth.
bit_depths = [3, 4, 6, 8, 12]
errors = {"MDCT per-band": [], "MDCT global": [], "Time-domain": []}
for b in bit_depths:
    errors["MDCT per-band"].append(report_error(x, mdct_synthesize(uniform_quantize(X_mdct, b, axis=0), length)))
    errors["MDCT global"].append(report_error(x, mdct_synthesize(uniform_quantize(X_mdct, b), length)))
    errors["Time-domain"].append(report_error(x, uniform_quantize(x, b)))

for name, values in errors.items():
    plt.semilogy(bit_depths, values, marker='o', label=name)
plt.xlabel("Bits per quantized value")
plt.ylabel("Relative RMS error")
plt.title("Effect of quantization depth (signal-dependent min/max ranges)")
plt.legend()
plt.grid(True, which='both', alpha=.3)
plt.show()


**Questions — Part 3**

1. Try 3, 4, 6 and 8 bits. Describe how distortion changes numerically and perceptually. Which sounds reveal quantization most clearly?
2. Compare global MDCT and per-bin MDCT quantization. Why can the result differ despite using the same nominal number of bits per coefficient? Which costs are omitted by this experiment?
3. Compare transform-domain and PCM quantization without assuming they have the same side information. Does lower waveform error always correspond to better listening quality?
4. In steady state, an MDCT produces approximately $f_s$ coefficients/s. Estimate raw index bitrate as

$$R_{\mathrm{payload}} \approx \frac{f_s b}{1000}\quad\mathrm{kb/s}.$$

For 16-bit mono PCM, use $R_{\mathrm{PCM}} = 16f_s/1000$ kb/s. Calculate both for your recording. Why are these **nominal payload** estimates rather than measured compressed-file bitrates?

*Write your answers below.*

## What to remember

- DFT and DCT-IV are different **invertible bases**, not compression methods on their own.
- STFT window length controls a practical time–frequency trade-off; hop size controls time sampling of frames.
- MDCT uses overlapping blocks but is **critically sampled in steady state**: $2M$ input samples per block, $M$ new input samples per hop, and $M$ real output coefficients.
- Proper MDCT windows and overlap-add cancel time-domain aliasing before quantization.
- Quantization reduces precision and introduces distortion; scaling, bit allocation, metadata and entropy coding matter for a practical bitrate.
- Listen as well as measuring error; do not confuse nominal coefficient count with compressed-file size.